# 03 · Chain-of-Thought Prompting
### Practical LLM Engineering — Module 02: Prompt Engineering

> **Objectives**
> - Why step-by-step reasoning dramatically improves LLM accuracy on hard tasks
> - Zero-shot CoT, few-shot CoT, and self-consistency decoding
> - How to implement and evaluate CoT pipelines
> - Tree of Thoughts and program-aided reasoning
> - When CoT helps — and when it hurts
> - Engineering tradeoffs: reasoning tokens, latency, and cost

---


## 1. Overview

**Chain-of-Thought (CoT) prompting** asks the model to produce intermediate reasoning steps before giving a final answer.

```
Standard prompting:
  Q: Roger has 5 tennis balls. He buys 2 more cans of 3 balls each.
     How many tennis balls does he have?
  A: 11   ← often wrong on harder problems

Chain-of-Thought:
  Q: Roger has 5 tennis balls. He buys 2 more cans of 3 balls each.
     How many tennis balls does he have? Think step by step.
  A: Roger starts with 5 balls.
     He buys 2 cans × 3 balls = 6 more balls.
     Total: 5 + 6 = 11 balls.
     The answer is 11.   ← correct, and verifiable
```

The reasoning trace serves two purposes:
1. **Computational** — gives the model "scratchpad space" to decompose hard problems
2. **Interpretive** — makes the reasoning transparent and auditable

### When CoT helps most

- Multi-step arithmetic and algebra
- Logical reasoning and constraint satisfaction
- Commonsense reasoning requiring world knowledge composition
- Code generation and debugging


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install anthropic openai transformers torch numpy matplotlib -q

import os, re, json, time, random, textwrap, math
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

print("✅ Libraries ready")


In [ ]:
# ── LLM client abstraction (from notebook 01) ────────────────────────
@dataclass
class LLMResponse:
    text: str; model: str; input_tokens: int
    output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class BaseLLMClient(ABC):
    @abstractmethod
    def complete(self, system, user, max_tokens=512, temperature=0.0) -> LLMResponse: ...
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

class ClaudeClient(BaseLLMClient):
    def __init__(self, model="claude-sonnet-4-20250514", api_key=None):
        import anthropic
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.messages.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{"role":"user","content":user}])
        return LLMResponse(msg.content[0].text, self.model,
            msg.usage.input_tokens, msg.usage.output_tokens,
            (time.perf_counter()-t0)*1000)

class OpenAIClient(BaseLLMClient):
    def __init__(self, model="gpt-4o-mini", api_key=None):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.chat.completions.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role":"system","content":system},{"role":"user","content":user}])
        c = msg.choices[0]
        return LLMResponse(c.message.content, self.model,
            msg.usage.prompt_tokens, msg.usage.completion_tokens,
            (time.perf_counter()-t0)*1000)

class HuggingFaceClient(BaseLLMClient):
    def __init__(self, model="microsoft/Phi-3-mini-4k-instruct", device="auto"):
        from transformers import pipeline
        self.model_name = model
        self.pipe = pipeline("text-generation", model=model,
                              device_map=device, torch_dtype="auto")
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        msgs = [{"role":"system","content":system},{"role":"user","content":user}]
        t0   = time.perf_counter()
        out  = self.pipe(msgs, max_new_tokens=max_tokens,
                         temperature=max(temperature,1e-4),
                         do_sample=temperature>0, return_full_text=False)
        text = out[0]["generated_text"]
        if isinstance(text, list): text = text[-1]["content"]
        return LLMResponse(text, self.model_name, 0,
                           len(text.split()), (time.perf_counter()-t0)*1000)

def get_llm(backend="claude", **kwargs):
    return {"claude":ClaudeClient,"anthropic":ClaudeClient,
            "openai":OpenAIClient,"gpt":OpenAIClient,
            "hf":HuggingFaceClient,"local":HuggingFaceClient}[backend.lower()](**kwargs)

# ── Mock client ───────────────────────────────────────────────────────
class MockLLMClient(BaseLLMClient):
    """Realistic mock responses for CoT demonstrations."""
    _MATH_COT = (
        "Let me work through this step by step.
"
        "First, I identify the key quantities.
"
        "Then I apply the relevant operations.
"
        "Finally, I compute: 5 + 2×3 = 5 + 6 = 11.
"
        "The answer is **11**."
    )
    _MATH_DIRECT = "11"
    _LOGIC_COT = (
        "Let me reason through the constraints.
"
        "Given: All A are B. Some B are C.
"
        "Therefore: Some A might be C, but we cannot conclude all A are C.
"
        "The answer is: Cannot be determined."
    )
    _GSME = (
        "Step 1: The store has 120 apples.
"
        "Step 2: 30% are sold → 120 × 0.30 = 36 apples sold.
"
        "Step 3: Remaining = 120 - 36 = 84 apples.
"
        "The answer is **84**."
    )
    _SC_ANSWERS = ["42", "42", "44", "42", "40"]  # for self-consistency demo
    _sc_idx = 0

    def complete(self, system, user, max_tokens=512, temperature=0.0):
        time.sleep(0.04)
        lower = user.lower()
        if "self-consistency" in system.lower() or temperature > 0.5:
            ans = self._SC_ANSWERS[self._sc_idx % len(self._SC_ANSWERS)]
            self.__class__._sc_idx += 1
            text = f"Working through this... the answer is {ans}."
        elif "step by step" in lower or "chain of thought" in lower or "reasoning" in lower:
            text = self._MATH_COT if any(c.isdigit() for c in user) else self._LOGIC_COT
        elif "direct" in system.lower() or "one word" in lower or "just the answer" in lower:
            text = self._MATH_DIRECT
        elif "apple" in lower or "store" in lower:
            text = self._GSME
        else:
            text = "After careful reasoning: the answer is 42."
        n_in = len((system + user).split())
        n_out = len(text.split())
        return LLMResponse(text, "mock-llm", n_in, n_out, 40.0)

llm = MockLLMClient()
print("MockLLMClient active — swap for get_llm('claude') when ready.")


## 3. Background: Why CoT Works

### 3.1 The scratchpad hypothesis

A transformer with $L$ layers and $d$-dimensional hidden states has a fixed computational budget per forward pass. For problems that require more than $L$ reasoning steps, the model cannot solve them in a single pass over the input tokens.

Chain-of-thought effectively **extends the computation graph** by letting the model use output tokens as intermediate variables:

$$
x_1, x_2, \ldots, x_n \xrightarrow{\text{generate}} r_1, r_2, \ldots, r_m \xrightarrow{\text{generate}} y
$$

where $r_i$ are reasoning tokens and $y$ is the final answer. Each reasoning token attends to all prior tokens, allowing multi-step deductions that a single forward pass cannot perform.

---

### 3.2 Theoretical grounding (Feng et al., 2023)

Transformers without scratchpad are limited to $\mathcal{TC}^0$ computations — they cannot solve problems that require unbounded serial reasoning (e.g. multi-digit multiplication, graph reachability).

With a scratchpad of $T$ reasoning tokens, the model can simulate $T$ computational steps, placing it in a strictly more powerful complexity class.

**Practical implication:** For problems requiring $k$ reasoning steps, you need at least $\mathcal{O}(k)$ thinking tokens.

---

### 3.3 Emergent behaviour at scale

Wei et al. (2022) showed CoT is an **emergent capability** — it only appears reliably in models with ~100B+ parameters. Smaller models often produce plausible-sounding but incorrect reasoning chains.

Modern instruction-tuned models of all sizes have improved significantly, but the principle still holds: CoT helps larger models more than smaller ones.


## 4. Zero-Shot CoT

The simplest form: just append **"Let's think step by step."** to the prompt.

Kojima et al. (2022) showed this single phrase dramatically improves accuracy on reasoning tasks — no examples needed.


In [ ]:
# ── Side-by-side: direct vs zero-shot CoT ─────────────────────────────
PROBLEMS = [
    {
        "question": (
            "A store has 120 apples. On Monday, 30% are sold. "
            "On Tuesday, 25% of the remaining apples are sold. "
            "How many apples are left?"
        ),
        "answer": "63"
    },
    {
        "question": (
            "If it takes 5 machines 5 minutes to make 5 widgets, "
            "how long does it take 100 machines to make 100 widgets?"
        ),
        "answer": "5"
    },
    {
        "question": (
            "There are 3 killers in a room. Someone enters and kills one of them. "
            "How many killers are in the room?"
        ),
        "answer": "3"
    },
]

SYSTEM = "You are a precise reasoning assistant."

def ask_direct(llm, question):
    return llm(system=SYSTEM,
               user=f"{question}\n\nProvide just the final answer.",
               max_tokens=20)

def ask_cot(llm, question):
    return llm(system=SYSTEM,
               user=f"{question}\n\nLet's think step by step.",
               max_tokens=300)

print("Zero-Shot CoT vs Direct Answering\n")
print("=" * 70)
for prob in PROBLEMS:
    q   = prob["question"]
    ans = prob["answer"]

    r_direct = ask_direct(llm, q)
    r_cot    = ask_cot(llm, q)

    print(f"Q: {textwrap.fill(q, 65, subsequent_indent='   ')}")
    print(f"Expected answer : {ans}")
    print(f"Direct          : {r_direct.text.strip()[:80]}  "
          f"({r_direct.output_tokens} tok)")
    print(f"CoT             : {r_cot.text.strip()[:120]}...")
    print(f"  (reasoning used {r_cot.output_tokens} tokens)")
    print()


In [ ]:
# ── Zero-shot CoT variants ────────────────────────────────────────────
# Different trigger phrases and their effect on reasoning quality
CoT_TRIGGERS = [
    "Let's think step by step.",
    "Think carefully and reason through this.",
    "Work through this problem systematically.",
    "First, let me identify what I know. Then I'll reason to the answer.",
    "Let me break this down into smaller parts.",
    "I'll solve this step by step, showing all my work.",
]

question = PROBLEMS[0]["question"]
print(f"Question: {question}\n")
print(f"{'Trigger':<48} {'Tokens':>7}  {'Answer excerpt'}")
print("-" * 80)

for trigger in CoT_TRIGGERS:
    resp = llm(system=SYSTEM,
               user=f"{question}\n\n{trigger}",
               max_tokens=200)
    last_line = [l.strip() for l in resp.text.strip().split("\n") if l.strip()][-1]
    print(f"{trigger[:46]:<48} {resp.output_tokens:>7}  {last_line[:40]}")


## 5. Few-Shot CoT

Provide complete reasoning traces as examples. The model learns **how** to reason, not just what to answer.


In [ ]:
# ── Few-shot CoT examples with full reasoning traces ─────────────────
FEW_SHOT_COT_EXAMPLES = [
    {
        "question": "Tom has 8 marbles. He gives half to Sarah, then buys 3 more. How many does he have?",
        "reasoning": (
            "Tom starts with 8 marbles.\n"
            "He gives half to Sarah: 8 ÷ 2 = 4 marbles given away.\n"
            "Tom now has 8 - 4 = 4 marbles.\n"
            "He buys 3 more: 4 + 3 = 7 marbles."
        ),
        "answer": "7"
    },
    {
        "question": "A train travels 60 km/h for 2 hours, then 80 km/h for 1.5 hours. What is the total distance?",
        "reasoning": (
            "First leg: 60 km/h × 2 h = 120 km.\n"
            "Second leg: 80 km/h × 1.5 h = 120 km.\n"
            "Total distance: 120 + 120 = 240 km."
        ),
        "answer": "240 km"
    },
    {
        "question": "A class of 30 students, 40% are girls. How many boys are there?",
        "reasoning": (
            "Girls: 40% of 30 = 0.40 × 30 = 12 girls.\n"
            "Boys: 30 - 12 = 18 boys."
        ),
        "answer": "18"
    },
]

def build_few_shot_cot_prompt(examples: list[dict], question: str) -> tuple[str, str]:
    """Build a few-shot CoT prompt with full reasoning traces."""
    system = (
        "You are a precise mathematical reasoning assistant. "
        "Always show your reasoning step by step before giving the final answer. "
        "End your response with 'The answer is <value>.'"
    )
    parts = ["Here are some worked examples:\n"]
    for i, ex in enumerate(examples, 1):
        parts.append(
            f"Example {i}:\n"
            f"Q: {ex['question']}\n"
            f"Reasoning:\n{ex['reasoning']}\n"
            f"The answer is {ex['answer']}.\n"
        )
    parts.append(f"Now solve this problem:\nQ: {question}\nReasoning:")
    return system, "\n".join(parts)

# Test it
TEST_Q = (
    "A recipe calls for 2.5 cups of flour for 12 cookies. "
    "How much flour is needed for 30 cookies?"
)

sys_p, usr_p = build_few_shot_cot_prompt(FEW_SHOT_COT_EXAMPLES, TEST_Q)
resp = llm(system=sys_p, user=usr_p, max_tokens=300)

print("Few-Shot CoT Reasoning Trace:")
print("-" * 50)
print(resp.text)
print(f"\n[{resp.output_tokens} reasoning tokens]")


In [ ]:
# ── Answer extraction utility ─────────────────────────────────────────
def extract_answer(text: str) -> Optional[str]:
    """
    Extract the final answer from a CoT response.
    Looks for patterns like 'The answer is X' or 'Answer: X'.
    """
    patterns = [
        r"[Tt]he answer is[:\s]+([\d,\.]+(?:\s*\w+)?)",
        r"[Aa]nswer[:\s]+([\d,\.]+(?:\s*\w+)?)",
        r"=\s*([\d,\.]+)\s*$",
        r"\*\*([\d,\.]+(?:\s*\w+)?)\*\*",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.MULTILINE)
        if m:
            return m.group(1).strip().rstrip(".")
    # Fallback: last number in text
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None


# Test extraction
samples = [
    "Step 1: ... Step 2: ... The answer is 42.",
    "After calculation: 5 + 6 = 11. **11**",
    "Total = 240 km",
    "Answer: 18 boys",
]
from typing import Optional
print("Answer extraction:")
for s in samples:
    print(f"  {s[:50]:<50} → {extract_answer(s)!r}")


## 6. Self-Consistency Decoding

A single CoT chain can make reasoning errors. **Self-consistency** (Wang et al., 2022) mitigates this by:

1. Sampling $N$ diverse reasoning paths (high temperature)
2. Extracting the final answer from each path
3. Taking the **majority vote** as the final answer

$$
\hat{y} = \underset{y}{\arg\max} \sum_{i=1}^{N} \mathbf{1}[\text{extract}(r_i) = y]
$$

where $r_i$ is the $i$-th reasoning chain.

**Why it works:** Different reasoning paths that arrive at the same answer are more likely to be correct than any single path. Errors tend to be idiosyncratic — they diversify over samples, while correct answers converge.

**Cost:** $N\times$ more LLM calls. Typical $N = 5$–$20$.


In [ ]:
def self_consistency(
    llm:         BaseLLMClient,
    system:      str,
    question:    str,
    n_samples:   int = 5,
    temperature: float = 0.7,
    max_tokens:  int = 300,
) -> dict:
    """
    Self-consistency decoding.
    Samples N reasoning paths and returns the majority-vote answer.
    """
    answers   = []
    traces    = []
    latencies = []
    tokens    = []

    for i in range(n_samples):
        resp = llm(
            system=system,
            user=f"{question}\n\nLet's think step by step.",
            max_tokens=max_tokens,
            temperature=temperature,
        )
        ans = extract_answer(resp.text) or resp.text.strip().split()[-1]
        answers.append(ans)
        traces.append(resp.text)
        latencies.append(resp.latency_ms)
        tokens.append(resp.total_tokens)

    # Majority vote
    vote_counts = Counter(answers)
    majority    = vote_counts.most_common(1)[0][0]
    confidence  = vote_counts[majority] / n_samples

    return {
        "majority_answer": majority,
        "confidence":      confidence,
        "vote_counts":     dict(vote_counts),
        "all_answers":     answers,
        "traces":          traces,
        "avg_tokens":      np.mean(tokens),
        "total_tokens":    sum(tokens),
        "avg_latency_ms":  np.mean(latencies),
    }


# ── Run self-consistency ──────────────────────────────────────────────
sc_question = (
    "A farmer has 17 sheep. All but 9 run away. "
    "How many sheep does the farmer have left?"
)
SYSTEM_MATH = (
    "You are a careful mathematical reasoning assistant. "
    "Show your work and end with 'The answer is <number>.'"
)

result = self_consistency(llm, SYSTEM_MATH, sc_question, n_samples=5, temperature=0.7)

print(f"Question  : {sc_question}")
print(f"Votes     : {result['vote_counts']}")
print(f"Majority  : {result['majority_answer']}  (confidence={result['confidence']:.0%})")
print(f"Tokens    : {result['total_tokens']} total across {len(result['all_answers'])} samples")
print(f"           (~{result['avg_tokens']:.0f} tokens/sample)")


In [ ]:
# ── Visualise self-consistency voting ────────────────────────────────
def plot_sc_votes(results_list: list[dict], questions: list[str]):
    fig, axes = plt.subplots(1, len(results_list), figsize=(5*len(results_list), 4))
    if len(results_list) == 1: axes = [axes]
    fig.suptitle("Self-Consistency: Vote Distribution", fontsize=12)

    for ax, res, q in zip(axes, results_list, questions):
        votes  = res["vote_counts"]
        labels = list(votes.keys())
        counts = list(votes.values())
        colors = ["#2ecc71" if l == res["majority_answer"] else "#95a5a6"
                  for l in labels]
        bars = ax.bar(labels, counts, color=colors, edgecolor="white")
        ax.set_title(f"Q: {q[:35]}...", fontsize=9)
        ax.set_ylabel("Votes")
        ax.set_xlabel("Answer")
        for bar, c in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    str(c), ha="center", fontsize=11, fontweight="bold")
        ax.set_ylim(0, max(counts) + 1)
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


# Run on two problems
problems_sc = [
    ("A farmer has 17 sheep. All but 9 run away. How many are left?", "9"),
    ("What is 15% of 240?", "36"),
]
sc_results, sc_qs = [], []
for q_text, _ in problems_sc:
    r = self_consistency(llm, SYSTEM_MATH, q_text, n_samples=5, temperature=0.7)
    sc_results.append(r)
    sc_qs.append(q_text)
    print(f"Q: {q_text}")
    print(f"   Majority={r['majority_answer']}  confidence={r['confidence']:.0%}  "
          f"votes={r['vote_counts']}")
    print()

plot_sc_votes(sc_results, sc_qs)


In [ ]:
# ── Self-consistency vs single CoT: accuracy comparison ──────────────
# Simulate accuracy over N samples showing convergence

np.random.seed(42)
N_TRIALS   = 20
TRUE_P_CORRECT = 0.65   # single-chain accuracy for this problem

# Single CoT: draw one answer
single_correct = np.random.binomial(1, TRUE_P_CORRECT, N_TRIALS)

# Self-consistency with k samples: majority vote correct if > k/2 correct
def sc_majority_correct(p, k, trials):
    results = []
    for _ in range(trials):
        votes = np.random.binomial(1, p, k)
        results.append(int(votes.sum() > k/2))
    return np.array(results)

k_values  = [1, 3, 5, 7, 10, 15, 20]
sc_accs   = [sc_majority_correct(TRUE_P_CORRECT, k, N_TRIALS).mean() for k in k_values]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_values, sc_accs, "o-", color="#3498db", lw=2, markersize=8, label="Self-consistency (majority vote)")
ax.axhline(TRUE_P_CORRECT, color="#e74c3c", linestyle="--", lw=1.5, label=f"Single CoT (p={TRUE_P_CORRECT})")
ax.axhline(1.0, color="gray", linestyle=":", lw=1, alpha=0.5)
ax.set_xlabel("Number of sampled reasoning paths (k)")
ax.set_ylabel("Expected accuracy")
ax.set_title("Self-Consistency: Accuracy improves with more samples")
ax.set_ylim(0.5, 1.05)
ax.legend()
ax.grid(alpha=0.3)

# Token cost axis
ax2 = ax.twinx()
avg_tok_per_sample = 150
ax2.plot(k_values, [k * avg_tok_per_sample for k in k_values],
         "s--", color="#e67e22", lw=1.5, alpha=0.6, label="Token cost")
ax2.set_ylabel("Total tokens (k × avg_per_sample)", color="#e67e22")
ax2.tick_params(axis="y", labelcolor="#e67e22")
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()


## 7. Structured CoT: Separating Reasoning from Answer

In production, you need to reliably extract the final answer from potentially long reasoning traces. A structured output format makes parsing deterministic.


In [ ]:
# ── Structured CoT with explicit reasoning / answer separation ────────
STRUCTURED_COT_SYSTEM = """You are a precise reasoning assistant.

When solving problems, always respond in this exact format:

<reasoning>
[Your step-by-step thinking here]
</reasoning>
<answer>
[Final answer only — a number, word, or short phrase]
</answer>"""


def parse_structured_cot(text: str) -> dict:
    """Parse structured CoT response into reasoning + answer components."""
    reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", text, re.DOTALL)
    answer_match    = re.search(r"<answer>(.*?)</answer>",    text, re.DOTALL)

    reasoning = reasoning_match.group(1).strip() if reasoning_match else ""
    answer    = answer_match.group(1).strip()    if answer_match    else extract_answer(text) or ""

    return {
        "reasoning":        reasoning,
        "answer":           answer,
        "reasoning_tokens": len(reasoning.split()),
        "well_formed":      bool(reasoning_match and answer_match),
    }


# ── Test on several problem types ─────────────────────────────────────
STRUCTURED_PROBLEMS = [
    "If a shirt costs $45 and is on sale for 20% off, what is the sale price?",
    "A rectangle has a perimeter of 28 cm and its length is twice its width. What is the area?",
    "John is older than Mary. Mary is older than Sue. Is John older than Sue?",
]

print("Structured CoT — reasoning/answer separation\n")
for q in STRUCTURED_PROBLEMS:
    resp   = llm(system=STRUCTURED_COT_SYSTEM, user=q, max_tokens=300)
    parsed = parse_structured_cot(resp.text)

    print(f"Q: {q}")
    print(f"  Reasoning ({parsed['reasoning_tokens']} words): "
          f"{parsed['reasoning'][:80]}...")
    print(f"  Answer   : {parsed['answer']}")
    print(f"  Parsed   : {'✅' if parsed['well_formed'] else '⚠️  malformed'}")
    print()


## 8. Program-Aided Language Models (PAL)

A powerful extension of CoT: instead of reasoning in natural language, the model writes **executable code** as the reasoning trace. The code is then run to get the exact answer.

```
Q: What is the compound interest on $1000 at 5% annually for 3 years?

CoT:  "1000 × 1.05 = 1050... then 1050 × 1.05..."  ← prone to arithmetic errors

PAL:  principal = 1000
      rate = 0.05
      years = 3
      amount = principal * (1 + rate) ** years
      answer = amount - principal   # → 157.625
```

PAL offloads exact computation to the Python interpreter, eliminating arithmetic errors entirely.


In [ ]:
PAL_SYSTEM = """You are a precise reasoning assistant that solves problems by writing Python code.

When given a problem:
1. Write clean Python code that solves it
2. Store the final answer in a variable called `answer`
3. Do NOT include print statements or imports (standard math is available)
4. Wrap the code in ```python ... ``` fences

Only output the code block, nothing else."""


def run_pal(llm: BaseLLMClient, question: str,
            allowed_builtins: dict = None) -> dict:
    """
    Program-Aided Language Model execution.
    Generates Python code with the LLM, then executes it safely.
    """
    resp = llm(system=PAL_SYSTEM, user=question, max_tokens=300)

    # Extract code from fences
    code_match = re.search(r"```python\n(.*?)```", resp.text, re.DOTALL)
    code_str   = code_match.group(1).strip() if code_match else resp.text.strip()

    # Safe execution namespace
    safe_ns = {"math": math, "__builtins__": {
        "abs": abs, "round": round, "int": int, "float": float,
        "min": min, "max": max, "sum": sum, "len": len, "range": range,
        "print": lambda *a: None,   # neutralise print
    }}
    if allowed_builtins:
        safe_ns.update(allowed_builtins)

    try:
        exec(code_str, safe_ns)
        answer = safe_ns.get("answer", "not found")
        error  = None
    except Exception as e:
        answer = None
        error  = str(e)

    return {
        "question":    question,
        "code":        code_str,
        "answer":      answer,
        "error":       error,
        "llm_tokens":  resp.total_tokens,
    }


# ── Demo PAL ──────────────────────────────────────────────────────────
PAL_PROBLEMS = [
    "What is 15% of 840?",
    "A car travels at 60 km/h for 2.5 hours. What distance does it cover?",
    "What is the compound interest on $2000 at 4% annual rate for 5 years?",
]

print("Program-Aided Language Models (PAL)\n")
for q in PAL_PROBLEMS:
    result = run_pal(llm, q)
    status = "✅" if result["error"] is None else f"❌ {result['error']}"
    print(f"Q: {q}")
    print(f"  Code   : {result['code'][:120].replace(chr(10), ' | ')}")
    print(f"  Answer : {result['answer']}  {status}")
    print()


## 9. Tree of Thoughts (ToT)

Chain-of-thought generates a single linear reasoning path. **Tree of Thoughts** (Yao et al., 2023) explores multiple reasoning branches simultaneously and uses a value function to select the most promising path.

```
Problem
├── Approach A
│   ├── Step A1 → [evaluate: promising ✅]
│   │   ├── Step A1a → answer
│   │   └── Step A1b → answer
│   └── Step A2 → [evaluate: dead end ❌]
└── Approach B
    └── Step B1 → [evaluate: weak ⚠️]
```

The key components:

1. **Thought generator** — produces $k$ candidate next steps
2. **State evaluator** — scores each partial solution (LLM-as-judge or heuristic)
3. **Search algorithm** — BFS or DFS over the thought tree

ToT is most useful for:
- Problems with clear intermediate checkpoints
- Tasks requiring backtracking (e.g. Sudoku, planning, creative writing)
- When a single reasoning path is reliably insufficient


In [ ]:
# ── Lightweight ToT implementation (BFS, depth-limited) ──────────────
@dataclass
class ThoughtNode:
    thought:    str
    score:      float = 0.0
    depth:      int   = 0
    parent:     Optional["ThoughtNode"] = None
    children:   list  = field(default_factory=list)

    def path(self) -> list[str]:
        """Return the full reasoning path from root to this node."""
        nodes, node = [], self
        while node:
            nodes.append(node.thought)
            node = node.parent
        return list(reversed(nodes))


def generate_thoughts(llm: BaseLLMClient, problem: str,
                       current_path: list[str], n: int = 3) -> list[str]:
    """Generate n candidate next reasoning steps."""
    path_str = "\n".join(f"  Step {i+1}: {s}" for i, s in enumerate(current_path))
    prompt   = (
        f"Problem: {problem}\n\n"
        f"Reasoning so far:\n{path_str if path_str else '  (start)'}\n\n"
        f"Generate {n} distinct candidate next steps. "
        f"Output exactly {n} lines, one step per line, numbered 1-{n}."
    )
    resp = llm(system="You are a systematic problem-solver.", user=prompt, max_tokens=200)
    lines = [re.sub(r"^\d+[.)\s]+", "", l).strip()
             for l in resp.text.strip().split("\n") if l.strip()]
    return lines[:n] if lines else ["Continue reasoning toward the answer."]


def evaluate_thought(llm: BaseLLMClient, problem: str,
                      path: list[str]) -> float:
    """Score a partial reasoning path on [0, 1]."""
    path_str = "\n".join(f"  {s}" for s in path)
    prompt   = (
        f"Problem: {problem}\n\n"
        f"Partial reasoning:\n{path_str}\n\n"
        f"Rate how likely this reasoning path leads to the correct answer.\n"
        f"Respond with a single number between 0.0 (wrong direction) and 1.0 (correct direction)."
    )
    resp  = llm(system="You are an evaluator.", user=prompt, max_tokens=10, temperature=0.0)
    match = re.search(r"0?\.\d+|1\.0|0|1", resp.text)
    return float(match.group()) if match else 0.5


def tree_of_thoughts_bfs(
    llm:       BaseLLMClient,
    problem:   str,
    breadth:   int = 3,    # candidates per node
    depth:     int = 3,    # max reasoning depth
    beam:      int = 2,    # keep top-k nodes per level
) -> ThoughtNode:
    """
    BFS Tree of Thoughts search.
    Returns the best leaf node found.
    """
    root    = ThoughtNode(thought=f"Problem: {problem}", depth=0)
    frontier = [root]

    for d in range(1, depth + 1):
        candidates = []
        for node in frontier:
            thoughts = generate_thoughts(llm, problem, node.path()[1:], n=breadth)
            for t in thoughts:
                child = ThoughtNode(thought=t, depth=d, parent=node)
                child.score = evaluate_thought(llm, problem, child.path()[1:])
                node.children.append(child)
                candidates.append(child)

        # Keep top `beam` nodes by score
        frontier = sorted(candidates, key=lambda x: x.score, reverse=True)[:beam]
        print(f"  Depth {d}: {len(candidates)} candidates → kept {len(frontier)} "
              f"(best score={frontier[0].score:.2f})")

    return frontier[0]   # best leaf


# ── Run ToT on a planning problem ─────────────────────────────────────
tot_problem = (
    "Plan how to efficiently pack a suitcase for a 5-day business trip "
    "that includes both formal meetings and casual dinners, given a weight limit of 10 kg."
)

print("Tree of Thoughts Search\n")
print(f"Problem: {tot_problem[:60]}...\n")
best_node = tree_of_thoughts_bfs(llm, tot_problem, breadth=2, depth=2, beam=2)

print("\nBest reasoning path:")
for i, step in enumerate(best_node.path()):
    print(f"  {'[Root]' if i==0 else f'Step {i}':>7}  {step[:80]}")
print(f"\nFinal score: {best_node.score:.2f}")


## 10. When CoT Hurts

CoT is not universally beneficial. Understanding when it degrades performance is as important as knowing when it helps.


In [ ]:
# ── Tasks where CoT hurts vs helps ──────────────────────────────────
tasks = {
    # CoT HELPS ──────────────────────────────────────────────────────
    "Multi-step arithmetic":    {
        "question": "What is (17 × 23) - (14 × 8) + 37?",
        "type": "helps",
        "reason": "Requires sequential computation; scratchpad prevents errors"
    },
    "Logical deduction":        {
        "question": "All mammals are warm-blooded. Dolphins are mammals. Are dolphins warm-blooded?",
        "type": "helps",
        "reason": "Benefits from explicit inference chain"
    },
    "Word problem":             {
        "question": "Alice is 3 years older than Bob. Bob is twice as old as Carol. Carol is 8. How old is Alice?",
        "type": "helps",
        "reason": "Requires multi-step variable tracking"
    },

    # CoT HURTS ───────────────────────────────────────────────────────
    "Simple factual":           {
        "question": "What is the capital of France?",
        "type": "hurts",
        "reason": "Direct recall; reasoning adds noise and tokens"
    },
    "Intuitive classification": {
        "question": "Is 'The food was amazing!' positive or negative?",
        "type": "hurts",
        "reason": "Pattern matching task; overthinking can introduce errors"
    },
    "Common knowledge":         {
        "question": "Is water wet?",
        "type": "hurts",
        "reason": "Generating reasoning can introduce unnecessary hedging"
    },
}

print(f"{'Task':<28} {'CoT Effect':<12} {'Reason'}")
print("-" * 80)
for name, info in tasks.items():
    effect = "✅ Helps" if info["type"] == "helps" else "⚠️  Hurts"
    print(f"{name:<28} {effect:<12} {info['reason']}")

print()
print("Rule of thumb: CoT helps when the task requires >2 serial reasoning steps.")
print("              CoT hurts when the answer is retrievable in one step.")


## 11. Evaluating CoT Quality

In [ ]:
# ── Benchmark: Direct vs CoT vs Self-Consistency ─────────────────────
BENCHMARK = [
    {"q": "A bag has 5 red, 3 blue, 2 green marbles. What fraction are red?",    "a": "1/2"},
    {"q": "Train A leaves at 9am at 60km/h. Train B at 10am at 80km/h. Same direction. When does B catch A?", "a": "1pm"},
    {"q": "What is 17% of 300?",                                                  "a": "51"},
    {"q": "A square has perimeter 24cm. What is its area?",                       "a": "36"},
    {"q": "If 6 workers build a wall in 12 days, how long for 9 workers?",        "a": "8"},
]

def run_benchmark(llm, problems, method="direct", n_sc=5):
    correct, token_counts, latencies = 0, [], []
    for prob in problems:
        q, expected = prob["q"], prob["a"]

        if method == "direct":
            resp = llm(system=SYSTEM_MATH, user=f"{q} Answer with just the value.",
                       max_tokens=20)
            pred = extract_answer(resp.text) or resp.text.strip()
            toks = resp.total_tokens

        elif method == "cot":
            resp = llm(system=STRUCTURED_COT_SYSTEM, user=q, max_tokens=300)
            parsed = parse_structured_cot(resp.text)
            pred = parsed["answer"]
            toks = resp.total_tokens

        elif method == "sc":
            result = self_consistency(llm, SYSTEM_MATH, q, n_samples=n_sc,
                                       temperature=0.7)
            pred = result["majority_answer"]
            toks = result["total_tokens"]

        # Fuzzy match: normalise answer strings
        def norm(s): return re.sub(r"[^0-9./]", "", str(s).lower())
        correct    += int(norm(pred) == norm(expected))
        token_counts.append(toks)

    return {
        "accuracy":   correct / len(problems),
        "avg_tokens": np.mean(token_counts),
        "total_tokens": sum(token_counts),
    }


methods = ["direct", "cot", "sc"]
bench_results = {}
print("Running benchmark...\n")
print(f"{'Method':<16} {'Accuracy':>10} {'Avg tokens':>12} {'Total tokens':>14}")
print("-" * 56)
for method in methods:
    res = run_benchmark(llm, BENCHMARK, method=method, n_sc=5)
    bench_results[method] = res
    print(f"{method:<16} {res['accuracy']:>10.2f} {res['avg_tokens']:>12.1f} "
          f"{res['total_tokens']:>14}")


In [ ]:
# ── Plot benchmark results ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Direct vs CoT vs Self-Consistency", fontsize=12)

method_labels  = ["Direct", "CoT", "Self-Consistency"]
method_keys    = ["direct", "cot", "sc"]
colors         = ["#95a5a6", "#3498db", "#2ecc71"]

accs   = [bench_results[m]["accuracy"]    for m in method_keys]
tokens = [bench_results[m]["avg_tokens"]  for m in method_keys]
# Efficiency: accuracy per 100 tokens
effs   = [a / (t / 100) if t > 0 else 0  for a, t in zip(accs, tokens)]

for ax, vals, ylabel, title in [
    (axes[0], accs,   "Accuracy",                "Task Accuracy"),
    (axes[1], tokens, "Avg tokens per problem",  "Token Cost"),
    (axes[2], effs,   "Accuracy per 100 tokens", "Token Efficiency"),
]:
    bars = ax.bar(method_labels, vals, color=colors, edgecolor="white", linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


## 12. Engineering Insights

### 12.1 Reasoning Token Budget

CoT significantly increases output token usage. In cost-sensitive systems:

$$
\text{cost increase} = \frac{T_{\text{reasoning}} \times p_{\text{out}}}{T_{\text{input}} \times p_{\text{in}}} \approx 3\text{–}10\times
$$

Output tokens are typically **3–5× more expensive** than input tokens.

### 12.2 Latency Impact

| Method | Relative latency | Notes |
|---|---|---|
| Direct | 1× | Short output, fast |
| Zero-shot CoT | 3–8× | Longer generation |
| Few-shot CoT | 3–8× + prefill | Extra input tokens |
| Self-consistency (k=5) | 15–40× | k parallel or serial calls |
| Tree of Thoughts | 10–50× | Many LLM calls |

### 12.3 Practical CoT Patterns for Production

```
Low latency required:
  → Direct or zero-shot CoT with short reasoning budget

High accuracy required, latency flexible:
  → Self-consistency with k=5–10

Verifiable / auditable answers needed:
  → Structured CoT with <reasoning>/<answer> tags

Arithmetic / calculation:
  → PAL (Python execution) — eliminates arithmetic errors entirely

Complex planning / multi-step:
  → Tree of Thoughts with beam=2–3, depth=3–4
```


In [ ]:
# ── Cost calculator: CoT reasoning budget ────────────────────────────
print("CoT cost analysis (Claude Sonnet, 50K calls/day)")
print()
p_in  = 3.0   # $/M input tokens
p_out = 15.0  # $/M output tokens

scenarios = {
    "Direct answer":           {"in": 50,  "out": 10},
    "Zero-shot CoT":           {"in": 50,  "out": 150},
    "Few-shot CoT (4 eg)":     {"in": 200, "out": 150},
    "Self-consistency k=5":    {"in": 50,  "out": 750},  # 5× CoT
    "PAL":                     {"in": 50,  "out": 80},   # code is concise
}

calls = 50_000
print(f"{'Scenario':<26} {'In tok':>8} {'Out tok':>8} {'$/day':>10} {'$/month':>10}")
print("-" * 68)
for name, toks in scenarios.items():
    cost_day   = calls * (toks["in"] * p_in + toks["out"] * p_out) / 1_000_000
    cost_month = cost_day * 30
    print(f"{name:<26} {toks['in']:>8} {toks['out']:>8} {cost_day:>10.2f} {cost_month:>10.2f}")


## 13. Exercises

1. **Trigger phrase sweep:** Test 10 different CoT trigger phrases ("Let's think step by step", "Work through this carefully", etc.) on 5 arithmetic problems. Does the exact phrasing matter, or does any step-by-step instruction work?

2. **Self-consistency calibration:** Run self-consistency with k ∈ {1, 3, 5, 10, 20} on 10 problems. Plot accuracy vs k. At what k does accuracy plateau? Is the confidence score (vote fraction) a good proxy for correctness?

3. **PAL vs CoT:** Take 10 arithmetic word problems and run both PAL and CoT. Which has higher accuracy? Which has lower token cost? In what situation would you choose each?

4. **Structured CoT parsing:** Build a robust `parse_structured_cot()` that handles malformed responses: reasoning without closing tag, answer embedded in reasoning, answer as an equation not a value. What fallbacks are needed?

5. **ToT evaluation:** Run Tree of Thoughts with different beam widths (beam=1, 2, 4) on a planning problem. Does a wider beam improve solution quality? At what beam does the cost become prohibitive?

---

## 14. Key Takeaways

| Concept | Summary |
|---|---|
| **Chain-of-Thought** | Reasoning tokens extend effective computation depth |
| **Zero-shot CoT** | "Let's think step by step" — free accuracy boost |
| **Few-shot CoT** | Full reasoning traces as examples — teaches reasoning style |
| **Self-consistency** | Majority vote over $k$ diverse paths — reliable accuracy |
| **PAL** | Code as scratchpad — eliminates arithmetic errors |
| **Tree of Thoughts** | BFS/DFS over reasoning branches — best for planning |
| **When CoT hurts** | Simple recall, pattern matching, single-step tasks |
| **Cost** | Output tokens expensive — budget reasoning length carefully |
| **Production pattern** | Structured `<reasoning>`/`<answer>` tags for reliable parsing |
